In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import mlflow.lightgbm
import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("✓ Imports complete")
print(f"  Optuna   : {optuna.__version__}")
print(f"  XGBoost  : {xgb.__version__}")
print(f"  LightGBM : {lgb.__version__}")

C:\Users\Mansoor\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
C:\Users\Mansoor\anaconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


✓ Imports complete
  Optuna   : 4.2.1
  XGBoost  : 2.0.3
  LightGBM : 4.3.0


In [2]:
df_train = pd.read_parquet(
    "../data/processed/featured/train_featured.parquet", engine="fastparquet"
)
df_val = pd.read_parquet(
    "../data/processed/featured/val_featured.parquet", engine="fastparquet"
)

with open("../data/processed/featured/feature_metadata.json") as f:
    feature_meta = json.load(f)

FEATURES = feature_meta["final_features"]
TARGET   = feature_meta["target"]

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_val   = df_val[FEATURES]
y_val   = df_val[TARGET]

print("✓ Data loaded")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")

✓ Data loaded
  X_train : (67493, 44)
  X_val   : (14473, 44)


In [3]:
EXPERIMENT_NAME = "procurement_ltv_tuning"
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)

def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    nonzero = y_true != 0
    mape = np.mean(
        np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])
    ) * 100
    dir_acc = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100
    return {
        "mae": round(mae, 4), "rmse": round(rmse, 4),
        "r2": round(r2, 4),   "mape": round(mape, 4),
        "dir_acc": round(dir_acc, 4)
    }

print("✓ MLflow experiment set")
print(f"  Experiment : {EXPERIMENT_NAME}")
print("✓ Metrics helper defined")

2026/05/08 10:02:49 INFO mlflow.tracking.fluent: Experiment with name 'procurement_ltv_tuning' does not exist. Creating a new experiment.


✓ MLflow experiment set
  Experiment : procurement_ltv_tuning
✓ Metrics helper defined


In [4]:
N_TRIALS_XGB = 50

def xgb_objective(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 300, 1000),
        "max_depth"        : trial.suggest_int("max_depth", 3, 8),
        "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight" : trial.suggest_int("min_child_weight", 5, 30),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "gamma"            : trial.suggest_float("gamma", 0.0, 2.0),
        "random_state"     : 42,
        "n_jobs"           : -1,
        "verbosity"        : 0,
        "eval_metric"      : "rmse",
        "early_stopping_rounds": 30,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print(f"Starting XGBoost Optuna tuning — {N_TRIALS_XGB} trials")
print("This will take ~5-8 minutes...\n")

xgb_study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=42),
    study_name="xgboost_tuning"
)
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)

print(f"\n✓ XGBoost tuning complete")
print(f"  Best trial       : #{xgb_study.best_trial.number}")
print(f"  Best Val MAE     : {xgb_study.best_value:.4f}")
print(f"\n  Best parameters:")
for k, v in xgb_study.best_params.items():
    print(f"    {k:<25} : {v}")

Starting XGBoost Optuna tuning — 50 trials
This will take ~5-8 minutes...



  0%|          | 0/50 [00:00<?, ?it/s]


✓ XGBoost tuning complete
  Best trial       : #23
  Best Val MAE     : 4.6649

  Best parameters:
    n_estimators              : 810
    max_depth                 : 8
    learning_rate             : 0.03521896484228155
    subsample                 : 0.895082200745162
    colsample_bytree          : 0.714026637043588
    min_child_weight          : 30
    reg_alpha                 : 0.25557146963654087
    reg_lambda                : 0.2127700389875931
    gamma                     : 0.6526020773817408


In [5]:
best_xgb_params = xgb_study.best_params.copy()
best_xgb_params.update({
    "random_state" : 42,
    "n_jobs"       : -1,
    "verbosity"    : 0,
    "eval_metric"  : "rmse",
    "early_stopping_rounds": 30,
})

with mlflow.start_run(run_name="xgboost_tuned"):
    best_xgb = xgb.XGBRegressor(**best_xgb_params)
    best_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    train_pred = best_xgb.predict(X_train)
    val_pred   = best_xgb.predict(X_val)

    train_m = compute_metrics(y_train, train_pred)
    val_m   = compute_metrics(y_val,   val_pred)

    log_params = {f"best_{k}": v for k, v in best_xgb_params.items()}
    log_params["best_iteration"]  = best_xgb.best_iteration
    log_params["n_optuna_trials"] = N_TRIALS_XGB
    log_params["model_type"]      = "XGBoost_tuned"

    mlflow.log_params(log_params)
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"val_{k}":   v for k, v in val_m.items()})
    mlflow.xgboost.log_model(best_xgb, "model")

print("BEST XGBOOST — TUNED RESULTS")
print(f"\n  Best iteration : {best_xgb.best_iteration}")
print(f"\n  {'Metric':<12} {'Baseline XGB':>14} {'Tuned XGB':>12} {'Improvement':>13}")
print("  " + "-" * 55)
baseline = {"mae":4.7010, "rmse":8.1193, "r2":0.4033, "dir_acc":91.52}
for metric in ["mae", "rmse", "r2", "dir_acc"]:
    base_val  = baseline[metric]
    tuned_val = val_m[metric]
    delta     = base_val - tuned_val if metric != "r2" else tuned_val - base_val
    symbol    = "▲ better" if delta > 0 else "▼ worse"
    print(f"  {metric:<12} {base_val:>14.4f} {tuned_val:>12.4f} "
          f"{abs(delta):>10.4f} {symbol}")

print(f"\n  Train MAE : {train_m['mae']:.4f}")
print(f"  Val MAE   : {val_m['mae']:.4f}")
print(f"  Overfit   : {train_m['mae'] - val_m['mae']:.4f} days")
print(f"\n  ✓ Logged to MLflow as 'xgboost_tuned'")

2026/05/08 10:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


BEST XGBOOST — TUNED RESULTS

  Best iteration : 653

  Metric         Baseline XGB    Tuned XGB   Improvement
  -------------------------------------------------------
  mae                  4.7010       4.6649     0.0361 ▲ better
  rmse                 8.1193       8.0872     0.0321 ▲ better
  r2                   0.4033       0.4080     0.0047 ▲ better
  dir_acc             91.5200      91.4876     0.0324 ▲ better

  Train MAE : 3.5347
  Val MAE   : 4.6649
  Overfit   : -1.1302 days

  ✓ Logged to MLflow as 'xgboost_tuned'


In [6]:
N_TRIALS_LGB = 50

def lgb_objective(trial):
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 300, 1500),
        "max_depth"         : trial.suggest_int("max_depth", 3, 8),
        "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves"        : trial.suggest_int("num_leaves", 20, 150),
        "subsample"         : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples" : trial.suggest_int("min_child_samples", 10, 50),
        "reg_alpha"         : trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda"        : trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "random_state"      : 42,
        "n_jobs"            : -1,
        "verbose"           : -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )
    preds = model.predict(X_val)
    return mean_absolute_error(y_val, preds)

print(f"Starting LightGBM Optuna tuning — {N_TRIALS_LGB} trials")
print("This will take ~5-8 minutes...\n")

lgb_study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=42),
    study_name="lightgbm_tuning"
)
lgb_study.optimize(lgb_objective, n_trials=N_TRIALS_LGB, show_progress_bar=True)

print(f"\n✓ LightGBM tuning complete")
print(f"  Best trial       : #{lgb_study.best_trial.number}")
print(f"  Best Val MAE     : {lgb_study.best_value:.4f}")
print(f"\n  Best parameters:")
for k, v in lgb_study.best_params.items():
    print(f"    {k:<25} : {v}")

Starting LightGBM Optuna tuning — 50 trials
This will take ~5-8 minutes...



  0%|          | 0/50 [00:00<?, ?it/s]


✓ LightGBM tuning complete
  Best trial       : #31
  Best Val MAE     : 4.6653

  Best parameters:
    n_estimators              : 1098
    max_depth                 : 8
    learning_rate             : 0.02666910874020058
    num_leaves                : 88
    subsample                 : 0.9420112478949296
    colsample_bytree          : 0.6268380003010464
    min_child_samples         : 45
    reg_alpha                 : 1.0097621621999935
    reg_lambda                : 0.00010137605763859207


In [7]:
best_lgb_params = lgb_study.best_params.copy()
best_lgb_params.update({
    "random_state" : 42,
    "n_jobs"       : -1,
    "verbose"      : -1,
})

with mlflow.start_run(run_name="lightgbm_tuned"):
    best_lgb = lgb.LGBMRegressor(**best_lgb_params)
    best_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(30, verbose=False),
            lgb.log_evaluation(period=-1)
        ]
    )

    train_pred = best_lgb.predict(X_train)
    val_pred   = best_lgb.predict(X_val)

    train_m = compute_metrics(y_train, train_pred)
    val_m   = compute_metrics(y_val,   val_pred)

    log_params = {f"best_{k}": v for k, v in best_lgb_params.items()}
    log_params["best_iteration"]  = best_lgb.best_iteration_
    log_params["n_optuna_trials"] = N_TRIALS_LGB
    log_params["model_type"]      = "LightGBM_tuned"

    mlflow.log_params(log_params)
    mlflow.log_metrics({f"train_{k}": v for k, v in train_m.items()})
    mlflow.log_metrics({f"val_{k}":   v for k, v in val_m.items()})
    mlflow.lightgbm.log_model(best_lgb, "model")

print("BEST LIGHTGBM — TUNED RESULTS")
print(f"\n  Best iteration : {best_lgb.best_iteration_}")
print(f"\n  {'Metric':<12} {'Baseline LGB':>14} {'Tuned LGB':>12} {'Improvement':>13}")
print("  " + "-" * 55)
baseline_lgb = {"mae":4.7091, "rmse":8.1174, "r2":0.4036, "dir_acc":91.48}
for metric in ["mae", "rmse", "r2", "dir_acc"]:
    base_val  = baseline_lgb[metric]
    tuned_val = val_m[metric]
    delta     = base_val - tuned_val if metric != "r2" else tuned_val - base_val
    symbol    = "▲ better" if delta > 0 else "▼ worse"
    print(f"  {metric:<12} {base_val:>14.4f} {tuned_val:>12.4f} "
          f"{abs(delta):>10.4f} {symbol}")

print(f"\n  Train MAE : {train_m['mae']:.4f}")
print(f"  Val MAE   : {val_m['mae']:.4f}")
print(f"  Overfit   : {train_m['mae'] - val_m['mae']:.4f} days")
print(f"\n  ✓ Logged to MLflow as 'lightgbm_tuned'")

2026/05/08 10:32:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 10:32:57 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


BEST LIGHTGBM — TUNED RESULTS

  Best iteration : 1048

  Metric         Baseline LGB    Tuned LGB   Improvement
  -------------------------------------------------------
  mae                  4.7091       4.6653     0.0438 ▲ better
  rmse                 8.1174       8.0780     0.0394 ▲ better
  r2                   0.4036       0.4094     0.0058 ▲ better
  dir_acc             91.4800      91.4392     0.0408 ▲ better

  Train MAE : 3.6578
  Val MAE   : 4.6653
  Overfit   : -1.0075 days

  ✓ Logged to MLflow as 'lightgbm_tuned'


In [8]:
print("=" * 65)
print("  HEAD-TO-HEAD: TUNED XGBOOST vs TUNED LIGHTGBM")
print("=" * 65)

xgb_val = compute_metrics(y_val, best_xgb.predict(X_val))
lgb_val = compute_metrics(y_val, best_lgb.predict(X_val))

print(f"\n  {'Metric':<12} {'XGBoost':>12} {'LightGBM':>12} {'Winner':>10}")
print("  " + "-" * 50)
for metric in ["mae", "rmse", "r2", "dir_acc"]:
    xv = xgb_val[metric]
    lv = lgb_val[metric]
    if metric in ["mae", "rmse"]:
        winner = "XGBoost" if xv < lv else "LightGBM"
    else:
        winner = "XGBoost" if xv > lv else "LightGBM"
    print(f"  {metric:<12} {xv:>12.4f} {lv:>12.4f} {winner:>10}")

print(f"\n  Decision: ", end="")
if xgb_val["mae"] < lgb_val["mae"]:
    print("XGBoost proceeds to Phase 10 as CHAMPION model")
    champion = "XGBoost"
else:
    print("LightGBM proceeds to Phase 10 as CHAMPION model")
    champion = "LightGBM"
print(f"  Champion : {champion}")

  HEAD-TO-HEAD: TUNED XGBOOST vs TUNED LIGHTGBM

  Metric            XGBoost     LightGBM     Winner
  --------------------------------------------------
  mae                4.6649       4.6653    XGBoost
  rmse               8.0872       8.0780   LightGBM
  r2                 0.4080       0.4094   LightGBM
  dir_acc           91.4876      91.4392    XGBoost

  Decision: XGBoost proceeds to Phase 10 as CHAMPION model
  Champion : XGBoost


In [9]:
import pickle

os.makedirs("../models", exist_ok=True)

# Save both tuned models + declare champion
if champion == "XGBoost":
    champion_model = best_xgb
    champion_params = best_xgb_params
    champion_name   = "xgboost_tuned"
else:
    champion_model = best_lgb
    champion_params = best_lgb_params
    champion_name   = "lightgbm_tuned"

# Save champion
with open(f"../models/champion_model.pkl", "wb") as f:
    pickle.dump(champion_model, f)

# Save both tuned models
with open("../models/xgboost_tuned.pkl", "wb") as f:
    pickle.dump(best_xgb, f)
with open("../models/lightgbm_tuned.pkl", "wb") as f:
    pickle.dump(best_lgb, f)

# Save tuning metadata
tuning_meta = {
    "champion"            : champion_name,
    "champion_val_mae"    : xgb_val["mae"] if champion == "XGBoost" else lgb_val["mae"],
    "xgb_best_params"     : {k: float(v) if isinstance(v, np.floating) else v
                              for k, v in best_xgb_params.items()},
    "lgb_best_params"     : {k: float(v) if isinstance(v, np.floating) else v
                              for k, v in best_lgb_params.items()},
    "xgb_best_iteration"  : int(best_xgb.best_iteration),
    "lgb_best_iteration"  : int(best_lgb.best_iteration_),
    "n_trials_xgb"        : N_TRIALS_XGB,
    "n_trials_lgb"        : N_TRIALS_LGB,
    "features"            : FEATURES,
    "target"              : TARGET,
}

with open("../models/tuning_metadata.json", "w") as f:
    json.dump(tuning_meta, f, indent=2)

print("✓ Models saved:")
for fname in ["champion_model.pkl", "xgboost_tuned.pkl", "lightgbm_tuned.pkl"]:
    size = os.path.getsize(f"../models/{fname}") / 1024 / 1024
    print(f"  {fname:<30} {size:.2f} MB")

print(f"\n✓ Tuning metadata saved to models/tuning_metadata.json")
print(f"\n  CHAMPION MODEL : {champion}")
print(f"  Val MAE        : {tuning_meta['champion_val_mae']:.4f} days")

✓ Models saved:
  champion_model.pkl             5.17 MB
  xgboost_tuned.pkl              5.17 MB
  lightgbm_tuned.pkl             6.87 MB

✓ Tuning metadata saved to models/tuning_metadata.json

  CHAMPION MODEL : XGBoost
  Val MAE        : 4.6649 days
